# Day 01 — Dynamic Arrays & Amortized Analysis

A dynamic array feels infinite — you keep appending and it grows. But memory is
fixed-size. The trick: when the block fills, allocate a **bigger** block and copy.
Grow by *doubling* and appends stay cheap on average. Let's *see* it.

## `size` vs `capacity`

- **size** (`len`): how many items you actually stored.
- **capacity**: how many slots you've allocated. Always `>= size`.

When `size == capacity`, the next append must resize.

In [ ]:
class DemoArray:
    """A doubling array, instrumented so we can watch it grow."""
    def __init__(self):
        self.cap = 1
        self.n = 0
        self.data = [None] * self.cap
        self.copies = 0          # total element-copies done by resizes

    def append(self, v):
        cost = 1                 # writing the new element is 1 unit of work
        if self.n == self.cap:   # full -> grow
            self.cap *= 2
            new = [None] * self.cap
            for i in range(self.n):
                new[i] = self.data[i]
                self.copies += 1
            cost += self.n       # ...plus the copies this append triggered
            self.data = new
        self.data[self.n] = v
        self.n += 1
        return cost

In [ ]:
a = DemoArray()
history = []                     # (index, size, capacity, cost_of_this_append)
for i in range(17):
    c = a.append(i)
    history.append((i, a.n, a.cap, c))

print(f"{'append#':>7} {'size':>5} {'capacity':>9} {'cost':>5}")
for idx, n, cap, cost in history:
    print(f'{idx:>7} {n:>5} {cap:>9} {cost:>5}')

## Capacity grows as a staircase of powers of two

Notice capacity jumps 1 → 2 → 4 → 8 → 16 → 32, doubling only when it has to.

In [ ]:
import matplotlib.pyplot as plt

idx = [h[0] for h in history]
size = [h[1] for h in history]
cap = [h[2] for h in history]

plt.figure(figsize=(7, 4))
plt.step(idx, cap, where='post', label='capacity (allocated)')
plt.plot(idx, size, marker='o', label='size (used)')
plt.xlabel('append #'); plt.ylabel('slots')
plt.title('Capacity doubles; size climbs by one')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Why it's *amortized O(1)*

Most appends cost 1. A few (at each resize) cost a lot. But because resizes get
**exponentially rarer**, the *average* cost per append stays flat and small.

In [ ]:
costs = [h[3] for h in history]
running_avg = [sum(costs[:k + 1]) / (k + 1) for k in range(len(costs))]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(idx, costs, alpha=0.4, label='cost of each append')
ax.plot(idx, running_avg, color='crimson', marker='o', label='running average cost')
ax.set_xlabel('append #'); ax.set_ylabel('work units')
ax.set_title('Spiky per-append cost, flat average → amortized O(1)')
ax.legend(); ax.grid(alpha=0.3); plt.show()

print('total copies for 17 appends:', a.copies)
print('average cost per append:', round(running_avg[-1], 2))

## Takeaways

- Doubling makes total copy-work across *n* appends only ~*2n* → **O(1) amortized**.
- Growing by a fixed `+k` instead would cost ~*n²/(2k)* → **O(n) per append**. Try it!

**Now go implement it:** open `homework.py`, then run `pytest -q`. The grader's
`test_amortized_constant_appends` fails loudly if you don't double.